# TabDiff on Kaggle: UNSW-NB15 and NSL-KDD submissions
Trains one TabDiff model per dataset (8000 epochs, paper default), samples the submissions and scores them locally.

Before running: attach the `tabdiff_comp` dataset, set **Accelerator = GPU T4 x2**, turn **Internet on**.
Use **Save Version -> Save & Run All (Commit)** so it keeps running after the browser closes (12 h limit).

In [ ]:
import glob, os, shutil, subprocess
src = os.path.dirname(glob.glob('/kaggle/input/**/TabDiff/main.py', recursive=True)[0])
WORK = '/kaggle/working/TabDiff'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print(os.getcwd(), os.listdir('.'))

In [ ]:
!pip install -q sdmetrics prdc icecream tomli==2.0.1 tomli-w category_encoders pyod
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Smoke test: 2 epochs, makes sure the environment works before the long run
!python main.py --dataname nslkdd --mode train --no_wandb --steps 2 --exp_name smoke --device cuda:0 2>&1 | tail -5
shutil.rmtree('tabdiff/ckpt/nslkdd/smoke', ignore_errors=True)
shutil.rmtree('tabdiff/result/nslkdd/smoke', ignore_errors=True)

In [ ]:
# Full training. With two GPUs both datasets train in parallel, otherwise one after the other.
import torch, time
os.makedirs('logs', exist_ok=True)
n_gpu = torch.cuda.device_count()
jobs = [('unsw', 'cuda:0'), ('nslkdd', 'cuda:1' if n_gpu > 1 else 'cuda:0')]

def launch(name, device):
    log = open(f'logs/train_{name}.log', 'w')
    return subprocess.Popen(
        ['python', 'main.py', '--dataname', name, '--mode', 'train', '--no_wandb',
         '--exp_name', 'baseline', '--device', device],
        stdout=log, stderr=subprocess.STDOUT, env={**os.environ, 'PYTHONUNBUFFERED': '1'})

def last_epoch(name):
    txt = open(f'logs/train_{name}.log').read().replace('\r', '\n')
    lines = [l for l in txt.split('\n') if l.startswith('Epoch ')]
    return lines[-1][:20] if lines else 'starting'

start = time.time()
if n_gpu > 1:
    procs = {name: launch(name, dev) for name, dev in jobs}
    while any(p.poll() is None for p in procs.values()):
        time.sleep(600)
        print(f'{(time.time()-start)/3600:.1f} h', {n: last_epoch(n) for n in procs}, flush=True)
    print({n: p.returncode for n, p in procs.items()})
else:
    for name, dev in jobs:
        p = launch(name, dev)
        while p.poll() is None:
            time.sleep(600)
            print(f'{(time.time()-start)/3600:.1f} h', name, last_epoch(name), flush=True)
        print(name, 'exit code', p.returncode)

In [ ]:
# Sample the two submissions and score them against the local proxy eval set
os.makedirs('submissions', exist_ok=True)
for name in ['nslkdd', 'unsw']:
    !python main.py --dataname {name} --mode sample --no_wandb --exp_name baseline --device cuda:0 --submission_path submissions/{name}.csv 2>&1 | grep -E "Submission|Error"
    !python comp/score.py --dataname {name} --real | tail -1
    !python comp/score.py --dataname {name} --submission submissions/{name}.csv | tail -1

In [ ]:
# Collect everything worth downloading into /kaggle/working/output
out = '/kaggle/working/output'
os.makedirs(out, exist_ok=True)
shutil.copytree('submissions', f'{out}/submissions', dirs_exist_ok=True)
shutil.copytree('logs', f'{out}/logs', dirs_exist_ok=True)
for name in ['unsw', 'nslkdd']:
    shutil.copytree(f'tabdiff/ckpt/{name}/baseline', f'{out}/ckpt/{name}/baseline', dirs_exist_ok=True)
!ls -R /kaggle/working/output | head -40